# CSE476 CA-1 Project: Build a Real Agent
## Topic T11: Smart Packing List Agent [Travel]

**Student Name / Mode**: Solo Project  
**Provider**: Groq (`qwen/qwen3.6-27b`)  
**Course**: CSE476 Agentic AI and Intelligent Automation  

---

### Agent Architecture Overview
This notebook demonstrates a **Plan-and-Act AI Agent** that dynamically prepares a customized packing list:
1. **Tool Use (2+ Tools)**: Dynamically calls `check_weather(city)` and `add_item(item, category, quantity)` / `add_items_batch(items)`.
2. **Plan-Act Loop**: Multi-step decision making — inspects weather observations and trip purpose to decide necessary gear.
3. **Session Memory**: Remembers destination, weather, and packed items across multiple conversational turns.

In [1]:
# Step 1: Initialize Agent and Reset State
import os
import json
from agent import SmartPackingAgent
from tools import clear_packing_list, get_packing_list

# Clear any previous session state
clear_packing_list()

# Instantiate our Plan-and-Act Agent
agent = SmartPackingAgent()
print("✅ SmartPackingAgent successfully initialized and ready!")

✅ SmartPackingAgent successfully initialized and ready!


---
### Scenario 1: Initial Goal (Plan & Act with Live Weather Lookup)
The user provides their destination (`Shimla`), duration (`3 days`), and purpose (`business conference`).  
**Expected Agentic Behavior**:
- Agent identifies destination and calls `check_weather('Shimla')`.
- Agent receives observation (Cold 9°C, Rainy 75% probability).
- Agent reasons: Cold + Rain + Work Conference $\rightarrow$ requires formal business attire, winter thermals, heavy jacket, umbrella, and laptop.
- Agent calls `add_item` / `add_items_batch` to populate the list.

In [2]:
turn1_goal = "I am going to Shimla for a 3-day business conference. What should I pack?"
result1 = agent.run_turn(turn1_goal, verbose=True)


👤 USER: I am going to Shimla for a 3-day business conference. What should I pack?



⚙️  [Step 1 - Action] Calling Tool: `check_weather`
    Arguments: {'city': 'Shimla'}
👁️  [Step 1 - Observation]:
    {
  "city": "Shimla, India",
  "temp_c": 9,
  "condition": "Cold & Rainy",
  "humidity": "82%",
  "rain_probability": "75%",
  "wind": "14 km/h",
  "clothing_hint": "Heavy winter layers, thermal innerwear, waterproof jacket / umbrella, wate... (truncated)



⚙️  [Step 2 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Clothing', 'item': 'Formal Suit', 'quantity': 1}
👁️  [Step 2 - Observation]:
    {"status": "added", "message": "Added 'Formal Suit' (qty: 1) to 'Clothing'.", "category": "Clothing", "item": "Formal Suit", "quantity": 1, "total_unique_items": 1}



⚙️  [Step 3 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Clothing', 'item': 'Thermal Innerwear', 'quantity': 2}
👁️  [Step 3 - Observation]:
    {"status": "added", "message": "Added 'Thermal Innerwear' (qty: 2) to 'Clothing'.", "category": "Clothing", "item": "Thermal Innerwear", "quantity": 2, "total_unique_items": 2}



⚙️  [Step 4 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Clothing', 'item': 'Heavy Winter Jacket', 'quantity': 1}
👁️  [Step 4 - Observation]:
    {"status": "added", "message": "Added 'Heavy Winter Jacket' (qty: 1) to 'Clothing'.", "category": "Clothing", "item": "Heavy Winter Jacket", "quantity": 1, "total_unique_items": 3}



⚙️  [Step 5 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Weather Gear', 'item': 'Umbrella', 'quantity': 1}
👁️  [Step 5 - Observation]:
    {"status": "added", "message": "Added 'Umbrella' (qty: 1) to 'Weather Gear'.", "category": "Weather Gear", "item": "Umbrella", "quantity": 1, "total_unique_items": 4}



⚙️  [Step 6 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Weather Gear', 'item': 'Waterproof Boots', 'quantity': 1}
👁️  [Step 6 - Observation]:
    {"status": "added", "message": "Added 'Waterproof Boots' (qty: 1) to 'Weather Gear'.", "category": "Weather Gear", "item": "Waterproof Boots", "quantity": 1, "total_unique_items": 5}



⚙️  [Step 7 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Clothing', 'item': 'Woolen Socks', 'quantity': 3}
👁️  [Step 7 - Observation]:
    {"status": "added", "message": "Added 'Woolen Socks' (qty: 3) to 'Clothing'.", "category": "Clothing", "item": "Woolen Socks", "quantity": 3, "total_unique_items": 6}



⚙️  [Step 8 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Work & Electronics', 'item': 'Laptop & Charger', 'quantity': 1}
👁️  [Step 8 - Observation]:
    {"status": "added", "message": "Added 'Laptop & Charger' (qty: 1) to 'Work & Electronics'.", "category": "Work & Electronics", "item": "Laptop & Charger", "quantity": 1, "total_unique_items": 7}


---
### Scenario 2: Conversational Memory Retrieval
The user asks a follow-up question without repeating all trip details:  
*"What weather gear do I have packed so far? Did you include an umbrella?"*  

**Expected Agentic Behavior**:
- Agent checks `SessionMemory` for previously added weather items.
- Answers accurately from memory without redundantly re-querying external APIs.

In [3]:
turn2_query = "What weather gear do I have packed so far? Did you include an umbrella?"
result2 = agent.run_turn(turn2_query, verbose=True)


👤 USER: What weather gear do I have packed so far? Did you include an umbrella?



⚙️  [Step 1 - Action] Calling Tool: `get_packing_list`
    Arguments: {}
👁️  [Step 1 - Observation]:
    {
  "total_items_count": 10,
  "categories": {
    "Clothing": [
      {
        "item": "Formal Suit",
        "quantity": 1
      },
      {
        "item": "Thermal Innerwear",
        "quantity": 2
      },
      {
        "item": "Heav... (truncated)



🤖 [Step 2 - Final Answer]:
Yes, I have included an **Umbrella** in your packing list. 

Here is the current **Weather Gear** section of your list:
*   **Umbrella** (1)
*   **Waterproof Boots** (1)

These were added because Shimla's forecast predicts cold, rainy conditions with a 75% chance of precipitation. Let me know if you need to add anything else!



---
### Scenario 3: Multi-Step Dynamic Re-Planning
The user introduces a new constraint:  
*"I decided to extend my stay in Shimla by 2 more days to go mountain trekking. What extra items should I pack?"*  

**Expected Agentic Behavior**:
- Agent retains Shimla context and existing packed clothes from memory.
- Agent reasons about trekking requirements (hiking boots, backpack, trekking pole, energy snacks, extra socks).
- Agent calls `add_item` / `add_items_batch` under the `Trekking & Outdoor` category to update the session packing list.

In [4]:
turn3_query = "I am extending the trip by 2 days to go mountain trekking. What extra items should I pack?"
result3 = agent.run_turn(turn3_query, verbose=True)


👤 USER: I am extending the trip by 2 days to go mountain trekking. What extra items should I pack?



⚙️  [Step 1 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Trekking & Outdoor', 'item': 'Hiking Boots', 'quantity': 1}
👁️  [Step 1 - Observation]:
    {"status": "added", "message": "Added 'Hiking Boots' (qty: 1) to 'Trekking & Outdoor'.", "category": "Trekking & Outdoor", "item": "Hiking Boots", "quantity": 1, "total_unique_items": 8}



⚙️  [Step 2 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Trekking & Outdoor', 'item': 'Rucksack', 'quantity': 1}
👁️  [Step 2 - Observation]:
    {"status": "added", "message": "Added 'Rucksack' (qty: 1) to 'Trekking & Outdoor'.", "category": "Trekking & Outdoor", "item": "Rucksack", "quantity": 1, "total_unique_items": 9}



⚙️  [Step 3 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Trekking & Outdoor', 'item': 'Trekking Poles', 'quantity': 1}
👁️  [Step 3 - Observation]:
    {"status": "added", "message": "Added 'Trekking Poles' (qty: 1) to 'Trekking & Outdoor'.", "category": "Trekking & Outdoor", "item": "Trekking Poles", "quantity": 1, "total_unique_items": 10}



⚙️  [Step 4 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Weather Gear', 'item': 'Rain Jacket', 'quantity': 1}
👁️  [Step 4 - Observation]:
    {"status": "added", "message": "Added 'Rain Jacket' (qty: 1) to 'Weather Gear'.", "category": "Weather Gear", "item": "Rain Jacket", "quantity": 1, "total_unique_items": 11}



⚙️  [Step 5 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Trekking & Outdoor', 'item': 'Water Bottle', 'quantity': 1}
👁️  [Step 5 - Observation]:
    {"status": "added", "message": "Added 'Water Bottle' (qty: 1) to 'Trekking & Outdoor'.", "category": "Trekking & Outdoor", "item": "Water Bottle", "quantity": 1, "total_unique_items": 12}



⚙️  [Step 6 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Clothing', 'item': 'Quick-dry T-shirts', 'quantity': 2}
👁️  [Step 6 - Observation]:
    {"status": "added", "message": "Added 'Quick-dry T-shirts' (qty: 2) to 'Clothing'.", "category": "Clothing", "item": "Quick-dry T-shirts", "quantity": 2, "total_unique_items": 13}



⚙️  [Step 7 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Clothing', 'item': 'Fleece Jacket', 'quantity': 1}
👁️  [Step 7 - Observation]:
    {"status": "added", "message": "Added 'Fleece Jacket' (qty: 1) to 'Clothing'.", "category": "Clothing", "item": "Fleece Jacket", "quantity": 1, "total_unique_items": 14}



⚙️  [Step 8 - Action] Calling Tool: `add_item`
    Arguments: {'category': 'Toiletries & Meds', 'item': 'Blister Plasters', 'quantity': 1}
👁️  [Step 8 - Observation]:
    {"status": "added", "message": "Added 'Blister Plasters' (qty: 1) to 'Toiletries & Meds'.", "category": "Toiletries & Meds", "item": "Blister Plasters", "quantity": 1, "total_unique_items": 15}


---
### Final Verified Packing State
Let's inspect the complete structured packing list built across all 3 conversational turns.

In [5]:
final_state = json.loads(get_packing_list())
print(f"📦 TOTAL PACKED ITEMS COUNT: {final_state['total_items_count']}\n")
for category, items in final_state['categories'].items():
    print(f"🔹 {category.upper()}:")
    for it in items:
        print(f"   • {it['item']} (Quantity: {it['quantity']})")
    print()

📦 TOTAL PACKED ITEMS COUNT: 19

🔹 CLOTHING:
   • Formal Suit (Quantity: 1)
   • Thermal Innerwear (Quantity: 2)
   • Heavy Winter Jacket (Quantity: 1)
   • Woolen Socks (Quantity: 3)
   • Quick-dry T-shirts (Quantity: 2)
   • Fleece Jacket (Quantity: 1)

🔹 WEATHER GEAR:
   • Umbrella (Quantity: 1)
   • Waterproof Boots (Quantity: 1)
   • Rain Jacket (Quantity: 1)

🔹 WORK & ELECTRONICS:
   • Laptop & Charger (Quantity: 1)

🔹 TREKKING & OUTDOOR:
   • Hiking Boots (Quantity: 1)
   • Rucksack (Quantity: 1)
   • Trekking Poles (Quantity: 1)
   • Water Bottle (Quantity: 1)

🔹 TOILETRIES & MEDS:
   • Blister Plasters (Quantity: 1)



---
### Viva Reference Summary
- **Plan-Act Decision Point**: Implemented in `agent.py` inside `SmartPackingAgent.run_turn()` within the `while step_count < max_steps:` loop.
- **Tool Calls**: `check_weather` and `add_item` / `add_items_batch` defined in `tools.py` and passed as OpenAI tool schemas.
- **Memory Integration**: Managed by `SessionMemory` in `memory.py`, which retains chat history, tool call IDs, tool observation messages, and trip context.